In [1]:
from pathlib import Path

print("Current working dir:")
print(Path.cwd())

Current working dir:
c:\Users\koushik\Desktop\variable-naming-service\app\services


In [14]:
from pathlib import Path
import json
from collections import defaultdict

# --- Detect project root dynamically ---
cwd = Path.cwd()

if (cwd / "data").exists():
    project_root = cwd
elif (cwd.parent / "data").exists():
    project_root = cwd.parent
elif (cwd.parent.parent / "data").exists():
    project_root = cwd.parent.parent
else:
    raise FileNotFoundError("Could not locate project root containing 'data' folder.")

print("Project root detected as:", project_root)

# --- JSON file paths ---
file1_path = project_root / "data" / "standards" / "autosar" / "abbreviation.json"
file2_path = project_root / "data" / "standards" / "autosar" / "new.json"

# --- Load JSON files ---
with open(file1_path, "r", encoding="utf-8") as f:
    json1 = json.load(f)

with open(file2_path, "r", encoding="utf-8") as f:
    json2 = json.load(f)

# ------------------------------------------------------------
# 1️⃣ Same keys but different values
# ------------------------------------------------------------
same_key_diff_value = {}

for key in json1.keys() & json2.keys():
    if json1[key] != json2[key]:
        same_key_diff_value[key] = {
            "file1": json1[key],
            "file2": json2[key]
        }

# ------------------------------------------------------------
# 2️⃣ Same values but different keys
# ------------------------------------------------------------
value_to_keys_1 = defaultdict(list)
value_to_keys_2 = defaultdict(list)

for k, v in json1.items():
    value_to_keys_1[v].append(k)

for k, v in json2.items():
    value_to_keys_2[v].append(k)

same_value_diff_keys = {}

common_values = set(value_to_keys_1.keys()) & set(value_to_keys_2.keys())

for value in common_values:
    keys1 = set(value_to_keys_1[value])
    keys2 = set(value_to_keys_2[value])
    if keys1 != keys2:
        same_value_diff_keys[value] = {
            "file1_keys": sorted(keys1),
            "file2_keys": sorted(keys2)
        }

print("\n=== SAME KEYS, DIFFERENT VALUES ===")
print(same_key_diff_value)
print(len(same_key_diff_value))

print("\n=== SAME VALUES, DIFFERENT KEYS ===")
print(len(same_value_diff_keys))
print(same_value_diff_keys)


Project root detected as: c:\Users\koushik\Desktop\variable-naming-service

=== SAME KEYS, DIFFERENT VALUES ===
{'timer': {'file1': 'Tmr', 'file2': 'tmr'}, 'type': {'file1': 'Typ', 'file2': 'TYP'}, 'interface': {'file1': 'Iface', 'file2': 'iface'}, 'vector': {'file1': 'Vec', 'file2': 'vec'}, 'function': {'file1': 'Func', 'file2': 'func'}, 'index': {'file1': 'Idx', 'file2': 'idx'}, 'control': {'file1': 'Ctl', 'file2': 'CTL'}, 'version': {'file1': 'Ver', 'file2': 'ver'}, 'KEEP': {'file1': 'Keep', 'file2': 'KEEP'}}
9

=== SAME VALUES, DIFFERENT KEYS ===
3
{'Inc': {'file1_keys': ['inclusion', 'increase'], 'file2_keys': ['increase']}, 'Sync': {'file1_keys': ['synchronization', 'synchronize'], 'file2_keys': ['synchronization']}, '2': {'file1_keys': ['2', 'to'], 'file2_keys': ['to']}}


In [16]:
from collections import defaultdict
import json

# ------------------------------------------------------------
# Work on a copy
# ------------------------------------------------------------
new_dict = json1.copy()

# ------------------------------------------------------------
# 1️⃣ Detect same abbreviation → different words
# ------------------------------------------------------------
value_to_keys = defaultdict(list)

for word, abbr in new_dict.items():
    value_to_keys[abbr].append(word)

same_abbr_conflicts = {
    abbr: words
    for abbr, words in value_to_keys.items()
    if len(words) > 1 and abbr != "__MISSING__"
}

print("\n=== SAME ABBREVIATION → DIFFERENT WORDS ===")
print("Count:", len(same_abbr_conflicts))
print(same_abbr_conflicts)


# ------------------------------------------------------------
# 2️⃣ Normalize abbreviation format
# Rule:
#   - Skip "__MISSING__"
#   - First letter uppercase
#   - Remaining lowercase
# ------------------------------------------------------------
format_fixed_count = 0

for word in new_dict:
    abbr = new_dict[word]

    if abbr == "__MISSING__":
        continue

    corrected = abbr.capitalize()

    if corrected != abbr:
        new_dict[word] = corrected
        format_fixed_count += 1

print("\nFormat corrections applied:", format_fixed_count)


# ------------------------------------------------------------
# 3️⃣ Sort dictionary by word ascending
# ------------------------------------------------------------
sorted_new_dict = dict(sorted(new_dict.items(), key=lambda x: x[0]))

print("Dictionary sorted ascending.")


# ------------------------------------------------------------
# 4️⃣ Save cleaned file
# ------------------------------------------------------------
output_path = project_root / "data" / "standards" / "autosar" / "abbreviation.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(sorted_new_dict, f, indent=2, ensure_ascii=False)

print("\nCleaned file saved to:", output_path)
print("Final size:", len(sorted_new_dict))


=== SAME ABBREVIATION → DIFFERENT WORDS ===
Count: 36
{'2': ['2', 'to'], 'Sum': ['addition', 'sum'], 'Circ': ['circle', 'circuit'], 'Cot': ['coated', 'cotangent'], 'Coll': ['collection', 'collector'], 'Cntr': ['container', 'counter'], 'Coord': ['coordinate', 'coordinated'], 'Deg': ['degree', 'degrees'], 'Dt': ['delta time', 'drivetrain'], 'Dep': ['dependency', 'depression'], 'Exp': ['expansion', 'exponential'], 'Ext': ['extension', 'external'], 'Fac': ['factor', 'factory'], 'Fw': ['file writer', 'freewheeling'], 'Gen': ['generation', 'generic'], 'Inc': ['inclusion', 'increase'], 'Lat': ['lateral', 'latitude'], 'Le': ['left', 'less or equal'], 'Mod': ['mode', 'modulo'], 'Opt': ['option', 'optional'], 'Pred': ['predication', 'predicted', 'prediction'], 'Prof': ['profile', 'profiler'], 'Rad': ['radian', 'radians'], 'Rel': ['relation', 'relative'], 'Rev': ['revision', 'revolution'], 'Sec': ['secant', 'second'], 'Sin': ['sine', 'sinus'], 'Soc': ['soc', 'stateofcharge'], 'Soe': ['soe', 'sta

In [3]:
# ------------------------------------------------------------
# 3️⃣ Simulate merge (old updated with new values)
# ------------------------------------------------------------
merged = json1.copy()
merged.update(json2)  # overwrite same keys with new values


# ------------------------------------------------------------
# 4️⃣ Recalculate SAME KEY DIFFERENT VALUE (should be 0)
# ------------------------------------------------------------
same_key_diff_after_merge = {}

for key in merged.keys() & json2.keys():
    if merged[key] != json2[key]:
        same_key_diff_after_merge[key] = {
            "merged": merged[key],
            "file2": json2[key]
        }

print("\n=== AFTER MERGE: SAME KEYS, DIFFERENT VALUES ===")
print(len(same_key_diff_after_merge))


# ------------------------------------------------------------
# 5️⃣ Recalculate SAME VALUE DIFFERENT KEYS
# ------------------------------------------------------------
from collections import defaultdict

value_to_keys_merged = defaultdict(list)
value_to_keys_2 = defaultdict(list)

for k, v in merged.items():
    value_to_keys_merged[v].append(k)

for k, v in json2.items():
    value_to_keys_2[v].append(k)

same_value_diff_keys_after_merge = {}

common_values = set(value_to_keys_merged.keys()) & set(value_to_keys_2.keys())

for value in common_values:
    keys1 = set(value_to_keys_merged[value])
    keys2 = set(value_to_keys_2[value])
    if keys1 != keys2:
        same_value_diff_keys_after_merge[value] = {
            "merged_keys": sorted(keys1),
            "file2_keys": sorted(keys2)
        }

print("\n=== AFTER MERGE: SAME VALUES, DIFFERENT KEYS ===")
print(len(same_value_diff_keys_after_merge))
print(same_value_diff_keys_after_merge)


=== AFTER MERGE: SAME KEYS, DIFFERENT VALUES ===
0

=== AFTER MERGE: SAME VALUES, DIFFERENT KEYS ===
6
{'Alt': {'merged_keys': ['alternative', 'alternator'], 'file2_keys': ['alternative']}, 'Out': {'merged_keys': ['out', 'output'], 'file2_keys': ['output']}, 'Sync': {'merged_keys': ['synchronization', 'synchronize'], 'file2_keys': ['synchronization']}, 'Dim': {'merged_keys': ['dim', 'dimension'], 'file2_keys': ['dimension']}, '2': {'merged_keys': ['2', 'to'], 'file2_keys': ['to']}, 'Auth': {'merged_keys': ['authentication', 'authorize'], 'file2_keys': ['authentication']}}


In [6]:
# ------------------------------------------------------------
# Controlled merge:
# 1) Overwrite old values with new ones
# 2) Remove keys where new value == "__MISSING__"
# ------------------------------------------------------------

merged = json1.copy()

removed_keys = []
updated_keys = []

for key, new_value in json2.items():

    # If new file says it's missing → remove from old
    if new_value == "__MISSING__":
        if key in merged:
            del merged[key]
            removed_keys.append(key)

    # Otherwise overwrite / insert
    else:
        if key in merged and merged[key] != new_value:
            updated_keys.append(key)
        merged[key] = new_value


print("\n=== MERGE SUMMARY ===")
print("Updated keys:", len(updated_keys))
print("Removed keys (because __MISSING__):", len(removed_keys))
print("Final dictionary size:", len(merged))


# ------------------------------------------------------------
# Optional: Save merged file
# ------------------------------------------------------------
output_path = project_root / "data" / "standards" / "autosar" / "abbreviation.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(merged, f, indent=2, ensure_ascii=False)

print("\nMerged file saved to:", output_path)


=== MERGE SUMMARY ===
Updated keys: 32
Removed keys (because __MISSING__): 30
Final dictionary size: 1415

Merged file saved to: c:\Users\koushik\Desktop\variable-naming-service\data\standards\autosar\abbreviation.json


In [16]:
%pip install pandas 

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ------------------ --------------------- 4.5/9.7 MB 36.3 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 37.8 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------- ----- 10.5/12.3 MB 54.8 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 48.4 MB/s  0:00:00

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Convert to DataFrame
import pandas as pd
rows = []

for word, values in same_key_diff_value.items():
    rows.append({
        "Word": word,
        "280 List": values["file1"],
        "Your JSON": values["file2"]
    })

df = pd.DataFrame(rows).sort_values("Word").reset_index(drop=True)

df.to_csv("conflicting.csv")


In [22]:
import json
from pathlib import Path

# --- Paths ---
project_root = Path("C:/Users/koushik/Desktop/variable-naming-service")
abbrev_file = project_root / "data" / "standards" / "autosar" / "abbreviation.json"
new_file = project_root / "data" / "standards" / "autosar" / "new.json"

# --- Load JSON files ---
with open(abbrev_file, "r", encoding="utf-8") as f:
    abbrev_data = json.load(f)

with open(new_file, "r", encoding="utf-8") as f:
    new_data = json.load(f)

# --- Merge: add missing keys from new.json to abbrev.json ---
for key, value in new_data.items():
    if key not in abbrev_data:
        abbrev_data[key] = value

# --- Sort by keys ---
abbrev_data_sorted = dict(sorted(abbrev_data.items()))

# --- Save back to abbreviation.json ---
with open(abbrev_file, "w", encoding="utf-8") as f:
    json.dump(abbrev_data_sorted, f, indent=4, ensure_ascii=False)

print(f"✅ Merged {len(abbrev_data_sorted) - len(abbrev_data)} new keys (if any) and saved sorted JSON.")


✅ Merged 0 new keys (if any) and saved sorted JSON.


In [28]:
import json
from pathlib import Path

# --- Paths ---
project_root = Path("C:/Users/koushik/Desktop/variable-naming-service")
abbrev_file = project_root / "data" / "standards" / "autosar" / "abbreviation.json"
new_file = project_root / "data" / "standards" / "autosar" / "new.json"
merged_file = project_root / "data" / "standards" / "autosar" / "abbreviation_merged.json"

# --- Load JSON files ---
with open(abbrev_file, "r", encoding="utf-8") as f:
    abbrev_data = json.load(f)

with open(new_file, "r", encoding="utf-8") as f:
    new_data = json.load(f)

# --- Existing keys and values ---
existing_keys = set(abbrev_data.keys())
existing_values = set(abbrev_data.values())

# --- Merge new keys/values only if key not exists and value not used ---
added_items = {}
debug_log = []

for key, value in new_data.items():
    if key in existing_keys:
        debug_log.append(f"SKIP: Key '{key}' already exists in abbreviation.json.")
        continue  # skip if key already exists
    if value in existing_values:
        debug_log.append(f"SKIP: Value '{value}' for key '{key}' already used by another key.")
        continue  # skip if value already used
    added_items[key] = value
    existing_keys.add(key)
    existing_values.add(value)
    debug_log.append(f"ADD: Key '{key}' with value '{value}' added to merged JSON.")

# --- Merge and sort ---
merged_data = {**abbrev_data, **added_items}
merged_data_sorted = dict(sorted(merged_data.items()))

# --- Save merged JSON ---
with open(merged_file, "w", encoding="utf-8") as f:
    json.dump(merged_data_sorted, f, indent=4, ensure_ascii=False)

# --- Print debug info ---
print(f"✅ Merged JSON created: {merged_file}")
print(f"Added {len(added_items)} new unique key-value pairs.\n")
print("=== DEBUG LOG ===")
for line in debug_log:
    print(line)


✅ Merged JSON created: C:\Users\koushik\Desktop\variable-naming-service\data\standards\autosar\abbreviation_merged.json
Added 0 new unique key-value pairs.

=== DEBUG LOG ===
SKIP: Key 'abbreviation' already exists in abbreviation.json.
SKIP: Key 'absolute' already exists in abbreviation.json.
SKIP: Key 'acronym' already exists in abbreviation.json.
SKIP: Key 'addition' already exists in abbreviation.json.
SKIP: Key 'address' already exists in abbreviation.json.
SKIP: Key 'algorithm' already exists in abbreviation.json.
SKIP: Key 'allocation' already exists in abbreviation.json.
SKIP: Key 'alternative' already exists in abbreviation.json.
SKIP: Key 'annotation' already exists in abbreviation.json.
SKIP: Key 'application' already exists in abbreviation.json.
SKIP: Key 'arccosecant' already exists in abbreviation.json.
SKIP: Key 'arccosine' already exists in abbreviation.json.
SKIP: Key 'arccotangent' already exists in abbreviation.json.
SKIP: Key 'arcsecant' already exists in abbreviati

In [29]:
# --- Paths ---
project_root = Path("C:/Users/koushik/Desktop/variable-naming-service")
abbrev_file = project_root / "data" / "standards" / "autosar" / "abbreviation.json"

# --- Load JSON ---
with open(abbrev_file, "r", encoding="utf-8") as f:
    abbrev_data = json.load(f)

# --- Detect value conflicts: same value for multiple keys ---
value_to_keys = defaultdict(list)
for key, value in abbrev_data.items():
    value_to_keys[value].append(key)

conflicting_values = {val: keys for val, keys in value_to_keys.items() if len(keys) > 1}

# --- Detect duplicate keys (in JSON this is unlikely, but checking) ---
# JSON cannot have duplicate keys when loaded into Python dict, so we can skip this
# If you read raw file and want to check duplicates, you can do:
raw_text = abbrev_file.read_text(encoding="utf-8")
lines = [line.strip() for line in raw_text.splitlines() if ":" in line]
key_counts = defaultdict(int)
for line in lines:
    k = line.split(":", 1)[0].strip().strip('"')
    key_counts[k] += 1

duplicate_keys = {k: c for k, c in key_counts.items() if c > 1}

# --- Print results ---
if conflicting_values:
    print("=== SAME VALUE, MULTIPLE KEYS ===")
    for val, keys in conflicting_values.items():
        print(f"Value '{val}' is used by keys: {keys}")
else:
    print("No value conflicts detected.")

if duplicate_keys:
    print("\n=== DUPLICATE KEYS ===")
    for key, count in duplicate_keys.items():
        print(f"Key '{key}' appears {count} times.")
else:
    print("\nNo duplicate keys detected.")

=== SAME VALUE, MULTIPLE KEYS ===
Value 'fp' is used by keys: ['file processor', 'function pointer']
Value 'pred' is used by keys: ['predication', 'prediction']
Value 'SOC' is used by keys: ['soc', 'stateofcharge']
Value 'SOE' is used by keys: ['soe', 'stateofenergy']
Value 'SOH' is used by keys: ['soh', 'stateofhealth']
Value 'SOHC' is used by keys: ['sohc', 'stateofhealthcapacity']

No duplicate keys detected.
